# Assignment Pipelines A and B: Shared SentencePiece, Pre-training, Fine-tuning, Evaluation

This combined notebook contains the full workflow for the two small T5 pipelines in the assignment:

1. Train or reuse one shared SentencePiece tokenizer on ~50K Java methods.
2. Pipeline A: pre-train T5-small with span corruption, then fine-tune on Java bug fixing.
3. Evaluate Pipeline A with exact match and CodeBLEU.
4. Pipeline B: skip pre-training, initialize a fresh T5-small with the same tokenizer/config, then fine-tune on the same bug-fixing task.
5. Evaluate Pipeline B with the same generation/evaluation code.

The main experimental rule is enforced here: Pipeline A and Pipeline B use the same SentencePiece tokenizer and the same fine-tuning hyperparameters. The only difference is whether the T5 weights were pre-trained first.

## 1. Configuration

For the final assignment run, keep `NUM_PRETRAIN_METHODS = 50000`, `PRETRAIN_EPOCHS = 3`, `VOCAB_SIZE = 16384`, and `CORRUPTION_RATE = 0.15`. The `FORCE_*` switches are false by default so completed work is reused.

In [2]:
import os
from pathlib import Path

ROOT = Path.cwd()

# Keep HuggingFace caches inside the project folder.
os.environ.setdefault("HF_HOME", str(ROOT / ".hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / ".hf_cache" / "transformers"))
os.environ.setdefault("HF_DATASETS_CACHE", str(ROOT / ".hf_cache" / "datasets"))

OUTPUT_DIR = ROOT / "outputs"
TOKENIZER_DIR = OUTPUT_DIR / "tokenizer"
TOKENIZER_PREFIX = TOKENIZER_DIR / "sp_code"
TOKENIZER_MODEL = TOKENIZER_DIR / "sp_code.model"
TOKENIZER_VOCAB = TOKENIZER_DIR / "sp_code.vocab"

PRETRAIN_CORPUS_FILE = OUTPUT_DIR / "corpus_java_50k.txt"
PRETRAIN_METHODS_FILE = OUTPUT_DIR / "methods_java_50k.jsonl"
PRETRAIN_TOKEN_IDS_FILE = OUTPUT_DIR / "token_ids_java_filtered.pt"
PIPELINE_A_PRETRAINED_DIR = OUTPUT_DIR / "pipeline_a_pretrained_model"
PIPELINE_A_PRETRAIN_CKPT_DIR = OUTPUT_DIR / "pipeline_a_checkpoints"
PIPELINE_A_PRETRAIN_LOG = OUTPUT_DIR / "pipeline_a_training_log.csv"

REFINE_TOKENIZED_CACHE = OUTPUT_DIR / "code_refinement_medium_tokenized.pt"
FINE_TUNE_HYPERPARAMS_FILE = OUTPUT_DIR / "fine_tuning_hyperparams.json"

PIPELINE_A_FINE_FINAL_DIR = OUTPUT_DIR / "pipeline_a_finetuned_model"
PIPELINE_A_FINE_BEST_DIR = OUTPUT_DIR / "pipeline_a_finetuned_best_model"
PIPELINE_A_FINE_CKPT_DIR = OUTPUT_DIR / "pipeline_a_finetune_checkpoints"
PIPELINE_A_FINE_LOG = OUTPUT_DIR / "pipeline_a_finetune_log.csv"
PIPELINE_A_PREDICTIONS = OUTPUT_DIR / "pipeline_a_test_predictions.jsonl"
PIPELINE_A_METRICS = OUTPUT_DIR / "pipeline_a_test_metrics.json"

PIPELINE_B_FINE_FINAL_DIR = OUTPUT_DIR / "pipeline_b_finetuned_model"
PIPELINE_B_FINE_BEST_DIR = OUTPUT_DIR / "pipeline_b_finetuned_best_model"
PIPELINE_B_FINE_CKPT_DIR = OUTPUT_DIR / "pipeline_b_finetune_checkpoints"
PIPELINE_B_FINE_LOG = OUTPUT_DIR / "pipeline_b_finetune_log.csv"
PIPELINE_B_PREDICTIONS = OUTPUT_DIR / "pipeline_b_test_predictions.jsonl"
PIPELINE_B_METRICS = OUTPUT_DIR / "pipeline_b_test_metrics.json"

for path in [
    OUTPUT_DIR, TOKENIZER_DIR, PIPELINE_A_PRETRAINED_DIR, PIPELINE_A_PRETRAIN_CKPT_DIR,
    PIPELINE_A_FINE_FINAL_DIR, PIPELINE_A_FINE_BEST_DIR, PIPELINE_A_FINE_CKPT_DIR,
    PIPELINE_B_FINE_FINAL_DIR, PIPELINE_B_FINE_BEST_DIR, PIPELINE_B_FINE_CKPT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
NUM_PRETRAIN_METHODS = 50_000
VOCAB_SIZE = 16_384
MIN_PRETRAIN_TOKENS = 10
MAX_PRETRAIN_LENGTH = 512
CORRUPTION_RATE = 0.15
PRETRAIN_EPOCHS = 3
PRETRAIN_BATCH_SIZE = 8
PRETRAIN_LR = 5e-4

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 512
FINE_TUNE_EPOCHS = 3
FINE_TUNE_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GEN_BATCH_SIZE = 8
FINE_TUNE_LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
GEN_MAX_LENGTH = 512
GEN_NUM_BEAMS = 4
EVAL_TEST_LIMIT = None  # set to a small integer only for a smoke test

STEP_CHECKPOINT_EVERY = 1000

FORCE_RETRAIN_TOKENIZER = False
FORCE_REPRETRAIN_PIPELINE_A = False
FORCE_REFIT_PIPELINE_A = False
FORCE_REFIT_PIPELINE_B = False
FORCE_REEVALUATE = False

print("Project root:", ROOT)
print("Shared tokenizer:", TOKENIZER_MODEL)

Project root: /home/apbasloe/CSCI555
Shared tokenizer: /home/apbasloe/CSCI555/outputs/tokenizer/sp_code.model


## 2. Imports, Device, and Shared Utilities

In [6]:
import csv
import json
import random
import shutil
import time
from collections import OrderedDict

import sentencepiece as spm
import torch
from codebleu import calc_codebleu
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import T5Config, T5ForConditionalGeneration

random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("torch:", torch.__version__)
print("torch CUDA build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def append_csv(path, header, row):
    write_header = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(header)
        writer.writerow(row)

def copy_tokenizer_into_model_dir(model_dir):
    tok_dir = model_dir / "tokenizer"
    tok_dir.mkdir(parents=True, exist_ok=True)
    for source in [TOKENIZER_MODEL, TOKENIZER_VOCAB]:
        if source.exists():
            shutil.copy2(source, tok_dir / source.name)

Using device: cuda
torch: 2.11.0+cu128
torch CUDA build: 12.8
cuda available: True
GPU: NVIDIA RTX A5000


## 3. Shared SentencePiece Tokenizer

SentencePiece is the tokenizer. It maps Java code text to token IDs. This tokenizer is trained once and then reused for Pipeline A and Pipeline B.

In [7]:
SENTINEL_TOKENS = [f"<extra_id_{i}>" for i in range(100)]

if PRETRAIN_METHODS_FILE.exists() and PRETRAIN_CORPUS_FILE.exists():
    pretrain_methods = [row["whole_func_string"] for row in read_jsonl(PRETRAIN_METHODS_FILE)]
    print(f"Loaded cached pre-training corpus: {len(pretrain_methods):,} methods")
else:
    print("Downloading/loading CodeSearchNet Java train split...")
    csn_train = load_dataset("code_search_net", "java", split="train")
    sampled = csn_train.shuffle(seed=SEED).select(range(NUM_PRETRAIN_METHODS))
    pretrain_methods = [row["whole_func_string"] for row in sampled if row.get("whole_func_string")]

    with PRETRAIN_METHODS_FILE.open("w", encoding="utf-8") as f_jsonl, PRETRAIN_CORPUS_FILE.open("w", encoding="utf-8") as f_txt:
        for method in pretrain_methods:
            f_jsonl.write(json.dumps({"whole_func_string": method}) + "\n")
            f_txt.write(method.replace("\n", " ") + "\n")
    print(f"Saved corpus files with {len(pretrain_methods):,} methods")

if TOKENIZER_MODEL.exists() and not FORCE_RETRAIN_TOKENIZER:
    print("Using existing shared SentencePiece tokenizer")
else:
    print("Training shared SentencePiece tokenizer...")
    spm.SentencePieceTrainer.train(
        input=str(PRETRAIN_CORPUS_FILE),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=VOCAB_SIZE,
        model_type="unigram",
        pad_id=0,
        eos_id=1,
        unk_id=2,
        bos_id=-1,
        user_defined_symbols=",".join(SENTINEL_TOKENS),
        character_coverage=1.0,
        hard_vocab_limit=False,
    )

sp = spm.SentencePieceProcessor()
sp.load(str(TOKENIZER_MODEL))

PAD_ID = sp.pad_id()
EOS_ID = sp.eos_id()
UNK_ID = sp.unk_id()
VOCAB_ACTUAL = sp.get_piece_size()

print("vocab size:", VOCAB_ACTUAL)
print("pad/eos/unk:", PAD_ID, EOS_ID, UNK_ID)
print("sentinel sanity:", [(tok, sp.piece_to_id(tok)) for tok in SENTINEL_TOKENS[:5]])
print("tokenization sanity:", sp.encode("public int add(int a, int b) { return a + b; }", out_type=str)[:30])

Loaded cached pre-training corpus: 50,000 methods
Using existing shared SentencePiece tokenizer
vocab size: 16384
pad/eos/unk: 0 1 2
sentinel sanity: [('<extra_id_0>', 3), ('<extra_id_1>', 4), ('<extra_id_2>', 5), ('<extra_id_3>', 6), ('<extra_id_4>', 7)]
tokenization sanity: ['▁public', '▁int', '▁add', '(', 'int', '▁a', ',', '▁int', '▁b', ')', '▁{', '▁return', '▁a', '▁+', '▁b', ';', '▁}']


## 4. Model Builder

This is the T5-small neural network. Pipeline A and Pipeline B use this same architecture. Pipeline A starts fine-tuning from pre-trained weights; Pipeline B starts from a fresh random initialization.

In [8]:
def build_t5_small_model():
    config = T5Config(
        decoder_start_token_id=PAD_ID,
        eos_token_id=EOS_ID,
        bos_token_id=PAD_ID,
        pad_token_id=PAD_ID,
        d_model=512,
        d_ff=2048,
        d_kv=64,
        num_heads=8,
        num_layers=6,
        num_decoder_layers=6,
        vocab_size=VOCAB_ACTUAL,
    )
    model = T5ForConditionalGeneration(config)
    model.resize_token_embeddings(VOCAB_ACTUAL)
    return model

def configure_special_tokens(model):
    model.config.pad_token_id = PAD_ID
    model.config.eos_token_id = EOS_ID
    model.config.bos_token_id = PAD_ID
    model.config.decoder_start_token_id = PAD_ID
    if model.config.vocab_size != VOCAB_ACTUAL:
        model.resize_token_embeddings(VOCAB_ACTUAL)
    return model

probe_model = build_t5_small_model()
print(f"Fresh T5-small parameters: {sum(p.numel() for p in probe_model.parameters()) / 1e6:.1f}M")
del probe_model

Fresh T5-small parameters: 52.4M


## 5. Pipeline A Pre-training Data and Span Corruption

Pipeline A uses span corruption on the ~50K Java-method corpus. This stage has no validation split, no early stopping, and no checkpoint selection. The final model after 3 epochs is used for fine-tuning.

In [9]:
def encode_pretrain_method(text):
    return sp.encode(text, out_type=int)[:MAX_PRETRAIN_LENGTH]

if PRETRAIN_TOKEN_IDS_FILE.exists():
    pretrain_token_ids = torch.load(PRETRAIN_TOKEN_IDS_FILE, weights_only=False)
    print(f"Loaded cached tokenized pre-training examples: {len(pretrain_token_ids):,}")
else:
    pretrain_token_ids = []
    for method in tqdm(pretrain_methods, desc="Tokenizing/filtering pre-train corpus"):
        ids = encode_pretrain_method(method)
        if MIN_PRETRAIN_TOKENS <= len(ids) <= MAX_PRETRAIN_LENGTH:
            pretrain_token_ids.append(ids)
    torch.save(pretrain_token_ids, PRETRAIN_TOKEN_IDS_FILE)
    print(f"Saved {len(pretrain_token_ids):,} tokenized pre-training examples")

def corrupt_tokens(tokens, corruption_rate=CORRUPTION_RATE):
    n_tokens = len(tokens)
    n_mask = min(n_tokens, max(1, int(n_tokens * corruption_rate)))
    mask_indices = sorted(random.sample(range(n_tokens), n_mask))

    spans = []
    current = [mask_indices[0]]
    for idx in mask_indices[1:]:
        if idx == current[-1] + 1:
            current.append(idx)
        else:
            spans.append(current)
            current = [idx]
    spans.append(current)

    if len(spans) + 1 > len(SENTINEL_TOKENS):
        spans = spans[: len(SENTINEL_TOKENS) - 1]

    sentinel_ids = [sp.piece_to_id(f"<extra_id_{i}>") for i in range(len(spans) + 1)]
    span_start_to_index = {span[0]: i for i, span in enumerate(spans)}
    masked_positions = {pos for span in spans for pos in span}

    input_ids = []
    i = 0
    while i < n_tokens:
        if i in span_start_to_index:
            span_index = span_start_to_index[i]
            input_ids.append(sentinel_ids[span_index])
            i = spans[span_index][-1] + 1
        elif i in masked_positions:
            i += 1
        else:
            input_ids.append(tokens[i])
            i += 1

    label_ids = []
    for span_index, span in enumerate(spans):
        label_ids.append(sentinel_ids[span_index])
        label_ids.extend(tokens[pos] for pos in span)
    label_ids.append(sentinel_ids[len(spans)])
    return input_ids, label_ids

class SpanCorruptionDataset(Dataset):
    def __init__(self, encoded_methods):
        self.encoded_methods = encoded_methods

    def __len__(self):
        return len(self.encoded_methods)

    def __getitem__(self, idx):
        input_ids, labels = corrupt_tokens(self.encoded_methods[idx])
        return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels)}

def collate_pretrain(batch):
    max_input_len = max(item["input_ids"].numel() for item in batch)
    max_label_len = max(item["labels"].numel() for item in batch)
    input_ids = torch.full((len(batch), max_input_len), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_input_len), dtype=torch.long)
    labels = torch.full((len(batch), max_label_len), -100, dtype=torch.long)
    for row, item in enumerate(batch):
        src = item["input_ids"]
        tgt = item["labels"]
        input_ids[row, : src.numel()] = src
        attention_mask[row, : src.numel()] = 1
        labels[row, : tgt.numel()] = tgt
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

pretrain_dataset = SpanCorruptionDataset(pretrain_token_ids)
print(f"Pipeline A pre-training examples: {len(pretrain_dataset):,}")

Loaded cached tokenized pre-training examples: 48,055
Pipeline A pre-training examples: 48,055


## 6. Run or Reuse Pipeline A Pre-training

In [10]:
def save_training_checkpoint(model, optimizer, checkpoint_dir, state_name, state_file_name, state):
    path = checkpoint_dir / state_name
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    torch.save({**state, "optimizer": optimizer.state_dict()}, path / state_file_name)

def checkpoint_candidates(checkpoint_dir, state_file_name):
    candidates = []
    for path in checkpoint_dir.glob("*"):
        state_file = path / state_file_name
        if state_file.exists():
            state = torch.load(state_file, map_location="cpu", weights_only=False)
            candidates.append((state.get("global_step", 0), str(path), path, state))
    return sorted(candidates, key=lambda item: (item[0], item[1]))

def move_optimizer_to_device(optimizer):
    for opt_state in optimizer.state.values():
        for key, value in opt_state.items():
            if torch.is_tensor(value):
                opt_state[key] = value.to(DEVICE)

def rng_state_dict():
    state = {"python_random_state": random.getstate(), "torch_rng_state": torch.get_rng_state()}
    if DEVICE.type == "cuda":
        state["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if "python_random_state" in state:
        random.setstate(state["python_random_state"])
    if "torch_rng_state" in state:
        torch.set_rng_state(state["torch_rng_state"])
    if DEVICE.type == "cuda" and "cuda_rng_state_all" in state:
        torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

def run_pipeline_a_pretraining():
    if (PIPELINE_A_PRETRAINED_DIR / "config.json").exists() and not FORCE_REPRETRAIN_PIPELINE_A:
        print("Pipeline A pre-trained model already exists; reusing:", PIPELINE_A_PRETRAINED_DIR)
        return

    latest = checkpoint_candidates(PIPELINE_A_PRETRAIN_CKPT_DIR, "training_state.pt")
    if latest:
        _, _, ckpt_path, resume_state = latest[-1]
        print("Resuming Pipeline A pre-training from", ckpt_path)
        model = configure_special_tokens(T5ForConditionalGeneration.from_pretrained(ckpt_path))
    else:
        resume_state = {"next_epoch": 0, "next_batch_in_epoch": 0, "global_step": 0, "running_loss": 0.0, "running_batches": 0}
        model = build_t5_small_model()
        print("Initialized fresh model for Pipeline A pre-training")

    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR)
    if latest and "optimizer" in resume_state:
        optimizer.load_state_dict(resume_state["optimizer"])
        move_optimizer_to_device(optimizer)
        restore_rng_state(resume_state)

    start_epoch = int(resume_state.get("next_epoch", 0))
    start_batch = int(resume_state.get("next_batch_in_epoch", 0))
    global_step = int(resume_state.get("global_step", 0))
    running_loss = float(resume_state.get("running_loss", 0.0))
    running_batches = int(resume_state.get("running_batches", 0))

    for epoch in range(start_epoch, PRETRAIN_EPOCHS):
        epoch_start = time.time()
        model.train()
        generator = torch.Generator()
        generator.manual_seed(SEED + epoch)
        loader = DataLoader(pretrain_dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_pretrain, generator=generator)
        iterator = iter(loader)

        if epoch == start_epoch and start_batch > 0:
            print(f"Skipping {start_batch:,} completed pre-training batches in epoch {epoch + 1}")
            for _ in range(start_batch):
                next(iterator)
            restore_rng_state(resume_state)
        else:
            start_batch = 0
            running_loss = 0.0
            running_batches = 0

        progress = tqdm(enumerate(iterator, start=start_batch), total=len(loader), initial=start_batch, desc=f"Pre-train epoch {epoch + 1}/{PRETRAIN_EPOCHS}")
        for batch_index, batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()

            running_loss += loss.item()
            running_batches += 1
            global_step += 1
            completed_batch = batch_index + 1
            progress.set_postfix(loss=f"{loss.item():.4f}", avg=f"{running_loss / running_batches:.4f}")

            if STEP_CHECKPOINT_EVERY and completed_batch % STEP_CHECKPOINT_EVERY == 0:
                state = {"next_epoch": epoch, "next_batch_in_epoch": completed_batch, "global_step": global_step, "running_loss": running_loss, "running_batches": running_batches, **rng_state_dict()}
                save_training_checkpoint(model, optimizer, PIPELINE_A_PRETRAIN_CKPT_DIR, "latest_step", "training_state.pt", state)

        avg_loss = running_loss / max(1, running_batches)
        elapsed = time.time() - epoch_start
        print(f"Pre-training epoch {epoch + 1} loss: {avg_loss:.4f} ({elapsed / 60:.1f} min)")
        append_csv(PIPELINE_A_PRETRAIN_LOG, ["epoch", "avg_loss", "seconds", "global_step"], [epoch + 1, avg_loss, round(elapsed, 2), global_step])
        state = {"next_epoch": epoch + 1, "next_batch_in_epoch": 0, "global_step": global_step, "running_loss": 0.0, "running_batches": 0, **rng_state_dict()}
        save_training_checkpoint(model, optimizer, PIPELINE_A_PRETRAIN_CKPT_DIR, f"epoch_{epoch + 1}", "training_state.pt", state)
        start_batch = 0
        running_loss = 0.0
        running_batches = 0

    print("Saving final Pipeline A pre-trained model")
    model.save_pretrained(PIPELINE_A_PRETRAINED_DIR)
    copy_tokenizer_into_model_dir(PIPELINE_A_PRETRAINED_DIR)

run_pipeline_a_pretraining()

Pipeline A pre-trained model already exists; reusing: /home/apbasloe/CSCI555/outputs/pipeline_a_pretrained_model


## 7. Fine-tuning Dataset: CodeXGLUE Code Refinement Medium

Input is `buggy`; target is `fixed`. This single tokenized cache is reused by Pipeline A and Pipeline B.

In [11]:
def encode_text(text, max_length):
    ids = sp.encode(text or "", out_type=int)
    ids = ids[: max_length - 1]
    ids.append(EOS_ID)
    return ids

def decode_ids(ids):
    clean = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id == EOS_ID:
            break
        if token_id in (PAD_ID, -100):
            continue
        clean.append(token_id)
    return sp.decode(clean)

def tokenize_refinement_split(split):
    rows = []
    for sample in tqdm(split, desc="Tokenizing CodeXGLUE split"):
        rows.append({
            "input_ids": encode_text(sample["buggy"], MAX_SOURCE_LENGTH),
            "labels": encode_text(sample["fixed"], MAX_TARGET_LENGTH),
            "buggy": sample["buggy"],
            "fixed": sample["fixed"],
        })
    return rows

fine_tuning_hyperparams = OrderedDict(
    dataset="google/code_x_glue_cc_code_refinement",
    dataset_name="medium",
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH,
    epochs=FINE_TUNE_EPOCHS,
    batch_size=FINE_TUNE_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=FINE_TUNE_LR,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    generation_max_length=GEN_MAX_LENGTH,
    generation_num_beams=GEN_NUM_BEAMS,
    seed=SEED,
)
write_json(FINE_TUNE_HYPERPARAMS_FILE, fine_tuning_hyperparams)

cache_meta = {"tokenizer_model": str(TOKENIZER_MODEL), "vocab_size": VOCAB_ACTUAL, "max_source_length": MAX_SOURCE_LENGTH, "max_target_length": MAX_TARGET_LENGTH}
if REFINE_TOKENIZED_CACHE.exists():
    cached = torch.load(REFINE_TOKENIZED_CACHE, weights_only=False)
    tokenized_refinement = cached["splits"] if cached.get("meta") == cache_meta else None
else:
    tokenized_refinement = None

if tokenized_refinement is None:
    raw_refinement = load_dataset("google/code_x_glue_cc_code_refinement", name="medium")
    val_name = "validation" if "validation" in raw_refinement else "valid"
    tokenized_refinement = {
        "train": tokenize_refinement_split(raw_refinement["train"]),
        "validation": tokenize_refinement_split(raw_refinement[val_name]),
        "test": tokenize_refinement_split(raw_refinement["test"]),
    }
    torch.save({"meta": cache_meta, "splits": tokenized_refinement}, REFINE_TOKENIZED_CACHE)
    print("Saved tokenized fine-tuning dataset")
else:
    print("Loaded cached tokenized fine-tuning dataset")

for split_name, rows in tokenized_refinement.items():
    print(f"{split_name}: {len(rows):,}")

Loaded cached tokenized fine-tuning dataset
train: 52,364
validation: 6,546
test: 6,545


## 8. Fine-tuning Helpers Shared by Pipeline A and Pipeline B

In [12]:
class CodeRefinementDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return {"input_ids": torch.tensor(row["input_ids"], dtype=torch.long), "labels": torch.tensor(row["labels"], dtype=torch.long)}

def collate_refinement(batch):
    max_input_len = max(item["input_ids"].numel() for item in batch)
    max_label_len = max(item["labels"].numel() for item in batch)
    input_ids = torch.full((len(batch), max_input_len), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_input_len), dtype=torch.long)
    labels = torch.full((len(batch), max_label_len), -100, dtype=torch.long)
    for row, item in enumerate(batch):
        src = item["input_ids"]
        tgt = item["labels"]
        input_ids[row, : src.numel()] = src
        attention_mask[row, : src.numel()] = 1
        labels[row, : tgt.numel()] = tgt
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = CodeRefinementDataset(tokenized_refinement["train"])
val_dataset = CodeRefinementDataset(tokenized_refinement["validation"])
test_rows = tokenized_refinement["test"]

def evaluate_loss(model, dataset):
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_refinement)
    model.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="validation", leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            total_loss += model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss.item()
            total_batches += 1
    model.train()
    return total_loss / max(1, total_batches)

def load_or_initialize_finetune_model(stage_name, initial_model_fn, final_dir, ckpt_dir, force_refit=False):
    if (final_dir / "config.json").exists() and not force_refit:
        print(f"{stage_name} final model already exists; loading {final_dir}")
        model = configure_special_tokens(T5ForConditionalGeneration.from_pretrained(final_dir))
        return model, None, {"next_epoch": FINE_TUNE_EPOCHS, "next_batch_in_epoch": 0, "global_step": 0, "best_val_loss": None}

    latest = checkpoint_candidates(ckpt_dir, "fine_tune_state.pt")
    if latest:
        _, _, ckpt_path, resume_state = latest[-1]
        print(f"Resuming {stage_name} from {ckpt_path}")
        model = configure_special_tokens(T5ForConditionalGeneration.from_pretrained(ckpt_path))
    else:
        resume_state = {"next_epoch": 0, "next_batch_in_epoch": 0, "global_step": 0, "best_val_loss": float("inf"), "running_loss": 0.0, "running_batches": 0}
        model = initial_model_fn()
        print(f"Initialized {stage_name}")

    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)
    if latest and "optimizer" in resume_state:
        optimizer.load_state_dict(resume_state["optimizer"])
        move_optimizer_to_device(optimizer)
        restore_rng_state(resume_state)
    return model, optimizer, resume_state

def fine_tune_stage(stage_name, initial_model_fn, final_dir, best_dir, ckpt_dir, log_file, force_refit=False):
    model, optimizer, resume_state = load_or_initialize_finetune_model(stage_name, initial_model_fn, final_dir, ckpt_dir, force_refit=force_refit)
    if int(resume_state.get("next_epoch", 0)) >= FINE_TUNE_EPOCHS and (final_dir / "config.json").exists() and not force_refit:
        print(f"{stage_name} fine-tuning already complete")
        return

    start_epoch = int(resume_state.get("next_epoch", 0))
    start_batch = int(resume_state.get("next_batch_in_epoch", 0))
    global_step = int(resume_state.get("global_step", 0))
    best_val_loss = float(resume_state.get("best_val_loss", float("inf")))
    running_loss = float(resume_state.get("running_loss", 0.0))
    running_batches = int(resume_state.get("running_batches", 0))

    for epoch in range(start_epoch, FINE_TUNE_EPOCHS):
        epoch_start = time.time()
        model.train()
        generator = torch.Generator()
        generator.manual_seed(SEED + epoch)
        loader = DataLoader(train_dataset, batch_size=FINE_TUNE_BATCH_SIZE, shuffle=True, collate_fn=collate_refinement, generator=generator)
        iterator = iter(loader)

        if epoch == start_epoch and start_batch > 0:
            print(f"Skipping {start_batch:,} completed batches in {stage_name} epoch {epoch + 1}")
            for _ in range(start_batch):
                next(iterator)
            restore_rng_state(resume_state)
        else:
            start_batch = 0
            running_loss = 0.0
            running_batches = 0

        progress = tqdm(enumerate(iterator, start=start_batch), total=len(loader), initial=start_batch, desc=f"{stage_name} epoch {epoch + 1}/{FINE_TUNE_EPOCHS}")
        for batch_index, batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()

            running_loss += loss.item()
            running_batches += 1
            global_step += 1
            completed_batch = batch_index + 1
            progress.set_postfix(loss=f"{loss.item():.4f}", avg=f"{running_loss / running_batches:.4f}")

            if STEP_CHECKPOINT_EVERY and completed_batch % STEP_CHECKPOINT_EVERY == 0:
                state = {"next_epoch": epoch, "next_batch_in_epoch": completed_batch, "global_step": global_step, "best_val_loss": best_val_loss, "running_loss": running_loss, "running_batches": running_batches, **rng_state_dict()}
                save_training_checkpoint(model, optimizer, ckpt_dir, "latest_step", "fine_tune_state.pt", state)

        train_loss = running_loss / max(1, running_batches)
        val_loss = evaluate_loss(model, val_dataset)
        elapsed = time.time() - epoch_start
        print(f"{stage_name} epoch {epoch + 1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, minutes={elapsed / 60:.1f}")
        append_csv(log_file, ["epoch", "train_loss", "val_loss", "seconds", "global_step"], [epoch + 1, train_loss, val_loss, round(elapsed, 2), global_step])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"Saving best-validation model for {stage_name}: {best_dir}")
            model.save_pretrained(best_dir)
            copy_tokenizer_into_model_dir(best_dir)

        state = {"next_epoch": epoch + 1, "next_batch_in_epoch": 0, "global_step": global_step, "best_val_loss": best_val_loss, "running_loss": 0.0, "running_batches": 0, **rng_state_dict()}
        save_training_checkpoint(model, optimizer, ckpt_dir, f"epoch_{epoch + 1}", "fine_tune_state.pt", state)
        start_batch = 0
        running_loss = 0.0
        running_batches = 0

    print(f"Saving final model for {stage_name}: {final_dir}")
    model.save_pretrained(final_dir)
    copy_tokenizer_into_model_dir(final_dir)

## 9. Pipeline A Fine-tuning

Pipeline A starts from `outputs/pipeline_a_pretrained_model`, which came from span-corruption pre-training.

In [9]:
def pipeline_a_initial_model():
    if not (PIPELINE_A_PRETRAINED_DIR / "config.json").exists():
        raise FileNotFoundError(f"Missing Pipeline A pre-trained model: {PIPELINE_A_PRETRAINED_DIR}")
    model = T5ForConditionalGeneration.from_pretrained(PIPELINE_A_PRETRAINED_DIR)
    return configure_special_tokens(model)

fine_tune_stage(
    "Pipeline A",
    pipeline_a_initial_model,
    PIPELINE_A_FINE_FINAL_DIR,
    PIPELINE_A_FINE_BEST_DIR,
    PIPELINE_A_FINE_CKPT_DIR,
    PIPELINE_A_FINE_LOG,
    force_refit=FORCE_REFIT_PIPELINE_A,
)

Initialized Pipeline A


Pipeline A epoch 1/3: 100%|██████████| 6546/6546 [09:00<00:00, 12.11it/s, avg=2.8447, loss=2.1637]  


Pipeline A epoch 1: train_loss=2.8447, val_loss=1.8627, minutes=9.3
Saving best-validation model for Pipeline A: /home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_best_model


Pipeline A epoch 2/3: 100%|██████████| 6546/6546 [09:00<00:00, 12.11it/s, avg=1.9154, loss=1.7995]  


Pipeline A epoch 2: train_loss=1.9154, val_loss=1.6515, minutes=9.3
Saving best-validation model for Pipeline A: /home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_best_model


Pipeline A epoch 3/3: 100%|██████████| 6546/6546 [09:00<00:00, 12.12it/s, avg=1.7472, loss=1.7851]  


Pipeline A epoch 3: train_loss=1.7472, val_loss=1.5579, minutes=9.3
Saving best-validation model for Pipeline A: /home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_best_model
Saving final model for Pipeline A: /home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_model


## 10. Evaluation Helper Shared by Pipeline A and Pipeline B

In [13]:
def generate_predictions(model_dir, rows, predictions_file, limit=EVAL_TEST_LIMIT):
    eval_rows = rows if limit is None else rows[:limit]
    model = configure_special_tokens(T5ForConditionalGeneration.from_pretrained(model_dir)).to(DEVICE)
    model.eval()

    sources, references, predictions = [], [], []
    for start in tqdm(range(0, len(eval_rows), GEN_BATCH_SIZE), desc=f"Generating {model_dir.name}"):
        batch_rows = eval_rows[start : start + GEN_BATCH_SIZE]
        max_len = max(len(row["input_ids"]) for row in batch_rows)
        input_ids = torch.full((len(batch_rows), max_len), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros((len(batch_rows), max_len), dtype=torch.long)
        for idx, row in enumerate(batch_rows):
            ids = torch.tensor(row["input_ids"], dtype=torch.long)
            input_ids[idx, : ids.numel()] = ids
            attention_mask[idx, : ids.numel()] = 1

        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids.to(DEVICE),
                attention_mask=attention_mask.to(DEVICE),
                max_length=GEN_MAX_LENGTH,
                num_beams=GEN_NUM_BEAMS,
                early_stopping=True,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                decoder_start_token_id=PAD_ID,
            )

        for row, generated_ids in zip(batch_rows, generated.cpu().tolist()):
            sources.append(row["buggy"].strip())
            references.append(row["fixed"].strip())
            predictions.append(decode_ids(generated_ids).strip())

    with predictions_file.open("w", encoding="utf-8") as f:
        for source, reference, prediction in zip(sources, references, predictions):
            f.write(json.dumps({"buggy": source, "fixed": reference, "prediction": prediction}) + "\n")
    return sources, references, predictions

def evaluate_stage(stage_name, final_dir, best_dir, predictions_file, metrics_file):
    if metrics_file.exists() and not FORCE_REEVALUATE:
        print(f"{stage_name} metrics already exist:")
        print(metrics_file.read_text(encoding="utf-8"))
        return json.loads(metrics_file.read_text(encoding="utf-8"))

    model_dir = best_dir if (best_dir / "config.json").exists() else final_dir
    if not (model_dir / "config.json").exists():
        raise FileNotFoundError(f"No model found for {stage_name}: {model_dir}")

    print(f"Evaluating {stage_name} with {model_dir}")
    sources, references, predictions = generate_predictions(model_dir, test_rows, predictions_file)
    exact_match_accuracy = sum(pred.strip() == ref.strip() for pred, ref in zip(predictions, references)) / max(1, len(predictions))
    codebleu_scores = calc_codebleu(references, predictions, lang="java")
    metrics = {"stage": stage_name, "model_dir": str(model_dir), "num_examples": len(predictions), "exact_match_accuracy": exact_match_accuracy, **codebleu_scores}
    write_json(metrics_file, metrics)
    print(json.dumps(metrics, indent=2))
    return metrics

## 11. Pipeline A Evaluation

In [14]:
pipeline_a_metrics = evaluate_stage(
    "Pipeline A",
    PIPELINE_A_FINE_FINAL_DIR,
    PIPELINE_A_FINE_BEST_DIR,
    PIPELINE_A_PREDICTIONS,
    PIPELINE_A_METRICS,
)

Pipeline A metrics already exist:
{
  "stage": "Pipeline A",
  "model_dir": "/home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_best_model",
  "num_examples": 6545,
  "exact_match_accuracy": 0.0,
  "codebleu": 0.06280486603758192,
  "ngram_match_score": 0.01554901149421028,
  "weighted_ngram_match_score": 0.029754688650928483,
  "syntax_match_score": 0.040785427016796785,
  "dataflow_match_score": 0.16513033698839216
}


## 12. Pipeline B Fine-tuning: No Pre-training

Pipeline B deliberately skips span-corruption pre-training. It uses the same SentencePiece tokenizer, same T5-small architecture, and same fine-tuning hyperparameters, but starts from fresh random weights.

In [12]:
def pipeline_b_initial_model():
    return build_t5_small_model()

fine_tune_stage(
    "Pipeline B",
    pipeline_b_initial_model,
    PIPELINE_B_FINE_FINAL_DIR,
    PIPELINE_B_FINE_BEST_DIR,
    PIPELINE_B_FINE_CKPT_DIR,
    PIPELINE_B_FINE_LOG,
    force_refit=FORCE_REFIT_PIPELINE_B,
)

Initialized Pipeline B


Pipeline B epoch 1/3: 100%|██████████| 6546/6546 [08:49<00:00, 12.36it/s, avg=0.8840, loss=0.5107]  


Pipeline B epoch 1: train_loss=0.8840, val_loss=0.4737, minutes=9.1
Saving best-validation model for Pipeline B: /home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_best_model


Pipeline B epoch 2/3: 100%|██████████| 6546/6546 [08:51<00:00, 12.32it/s, avg=0.4662, loss=0.4822]  


Pipeline B epoch 2: train_loss=0.4662, val_loss=0.3856, minutes=9.2
Saving best-validation model for Pipeline B: /home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_best_model


Pipeline B epoch 3/3: 100%|██████████| 6546/6546 [08:51<00:00, 12.31it/s, avg=0.4007, loss=0.5533]  


Pipeline B epoch 3: train_loss=0.4007, val_loss=0.3470, minutes=9.2
Saving best-validation model for Pipeline B: /home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_best_model
Saving final model for Pipeline B: /home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_model


## 13. Pipeline B Evaluation

In [15]:
pipeline_b_metrics = evaluate_stage(
    "Pipeline B",
    PIPELINE_B_FINE_FINAL_DIR,
    PIPELINE_B_FINE_BEST_DIR,
    PIPELINE_B_PREDICTIONS,
    PIPELINE_B_METRICS,
)

Pipeline B metrics already exist:
{
  "stage": "Pipeline B",
  "model_dir": "/home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_best_model",
  "num_examples": 6545,
  "exact_match_accuracy": 0.0006111535523300229,
  "codebleu": 0.47037517854648747,
  "ngram_match_score": 0.47314989836164356,
  "weighted_ngram_match_score": 0.47622479159383097,
  "syntax_match_score": 0.46249112845990065,
  "dataflow_match_score": 0.46963489577057477
}


## 14. Compare Pipeline A and Pipeline B

Use these numbers in the report discussion about whether pre-training helped.

In [16]:
def load_metrics(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

a = load_metrics(PIPELINE_A_METRICS)
b = load_metrics(PIPELINE_B_METRICS)
print("Shared tokenizer:", TOKENIZER_MODEL)
print("Fine-tuning hyperparameters:", FINE_TUNE_HYPERPARAMS_FILE)
print()
if a:
    print("Pipeline A:", json.dumps(a, indent=2))
if b:
    print("Pipeline B:", json.dumps(b, indent=2))
if a and b:
    print()
    print("Delta CodeBLEU (A - B):", a["codebleu"] - b["codebleu"])
    print("Delta exact match (A - B):", a["exact_match_accuracy"] - b["exact_match_accuracy"])

Shared tokenizer: /home/apbasloe/CSCI555/outputs/tokenizer/sp_code.model
Fine-tuning hyperparameters: /home/apbasloe/CSCI555/outputs/fine_tuning_hyperparams.json

Pipeline A: {
  "stage": "Pipeline A",
  "model_dir": "/home/apbasloe/CSCI555/outputs/pipeline_a_finetuned_best_model",
  "num_examples": 6545,
  "exact_match_accuracy": 0.0,
  "codebleu": 0.06280486603758192,
  "ngram_match_score": 0.01554901149421028,
  "weighted_ngram_match_score": 0.029754688650928483,
  "syntax_match_score": 0.040785427016796785,
  "dataflow_match_score": 0.16513033698839216
}
Pipeline B: {
  "stage": "Pipeline B",
  "model_dir": "/home/apbasloe/CSCI555/outputs/pipeline_b_finetuned_best_model",
  "num_examples": 6545,
  "exact_match_accuracy": 0.0006111535523300229,
  "codebleu": 0.47037517854648747,
  "ngram_match_score": 0.47314989836164356,
  "weighted_ngram_match_score": 0.47622479159383097,
  "syntax_match_score": 0.46249112845990065,
  "dataflow_match_score": 0.46963489577057477
}

Delta CodeBLEU (

## 15. RAG Comparison: Setup and Retriever Choice

This section implements the final assignment comparison using `Qwen/Qwen2.5-Coder-1.5B-Instruct` as the generator.

Retriever choice: CodeBERT mean-pooled embeddings over the buggy methods in the bug-fixing training set. This is a reasonable choice because CodeBERT was trained on source code, the query and knowledge base are both Java-like buggy methods, and embedding retrieval requires no extra supervised training. The retrieved examples are used as 3-shot context for Qwen.

The knowledge base is the same CodeXGLUE training split used for fine-tuning: `buggy -> fixed` pairs. The test set is the same test split used for Pipeline A and Pipeline B.

In [17]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

RAG_DIR = OUTPUT_DIR / "rag"
RAG_DIR.mkdir(parents=True, exist_ok=True)

CODEBERT_MODEL_NAME = "microsoft/codebert-base"
QWEN_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

RAG_TOP_K = 3
RAG_RETRIEVER_MAX_LENGTH = 256
RAG_EMBED_BATCH_SIZE = 32
QWEN_GENERATION_BATCH_SIZE = 1  # keep at 1 for predictable memory with long few-shot prompts
QWEN_MAX_NEW_TOKENS = 512
RAG_EVAL_LIMIT = None  # set to a small integer for smoke testing; None for final assignment numbers

FORCE_REBUILD_RAG_EMBEDDINGS = False
FORCE_REBUILD_RAG_RETRIEVAL = False
FORCE_REGENERATE_QWEN = False
FORCE_REEVALUATE_QWEN = False

RAG_TRAIN_EMBEDDINGS = RAG_DIR / "codebert_train_buggy_embeddings.pt"
RAG_TEST_EMBEDDINGS = RAG_DIR / "codebert_test_buggy_embeddings.pt"
RAG_TOPK_INDICES = RAG_DIR / "codebert_topk_train_indices.pt"

QWEN_ZERO_PREDICTIONS = RAG_DIR / "qwen_zero_shot_predictions.jsonl"
QWEN_ZERO_METRICS = RAG_DIR / "qwen_zero_shot_metrics.json"
QWEN_RAG_PREDICTIONS = RAG_DIR / "qwen_rag_3shot_predictions.jsonl"
QWEN_RAG_METRICS = RAG_DIR / "qwen_rag_3shot_metrics.json"

print("RAG directory:", RAG_DIR)
print("Retriever:", CODEBERT_MODEL_NAME)
print("Generator:", QWEN_MODEL_NAME)

RAG directory: /home/apbasloe/CSCI555/outputs/rag
Retriever: microsoft/codebert-base
Generator: Qwen/Qwen2.5-Coder-1.5B-Instruct


## 16. Build or Load CodeBERT Embeddings

This embeds the training-set buggy methods as the knowledge base and the test-set buggy methods as retrieval queries. Embeddings are normalized so cosine similarity is just a dot product.

In [18]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def embed_codebert_texts(texts, tokenizer, model, batch_size=RAG_EMBED_BATCH_SIZE):
    all_embeddings = []
    model.eval()
    for start in tqdm(range(0, len(texts), batch_size), desc="CodeBERT embeddings"):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=RAG_RETRIEVER_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).float()

def rag_embedding_meta():
    return {
        "codebert_model": CODEBERT_MODEL_NAME,
        "retriever_max_length": RAG_RETRIEVER_MAX_LENGTH,
        "train_count": len(tokenized_refinement["train"]),
        "test_count": len(test_rows),
    }

def cache_matches(path, meta):
    if not path.exists():
        return False
    try:
        cached = torch.load(path, map_location="cpu", weights_only=False)
        return cached.get("meta") == meta and "embeddings" in cached
    except Exception:
        return False

meta = rag_embedding_meta()
need_embeddings = (
    FORCE_REBUILD_RAG_EMBEDDINGS
    or not cache_matches(RAG_TRAIN_EMBEDDINGS, meta)
    or not cache_matches(RAG_TEST_EMBEDDINGS, meta)
)

if need_embeddings:
    print("Building CodeBERT embedding caches...")
    codebert_tokenizer = AutoTokenizer.from_pretrained(CODEBERT_MODEL_NAME)
    codebert_model = AutoModel.from_pretrained(CODEBERT_MODEL_NAME).to(DEVICE)

    train_buggy_texts = [row["buggy"] for row in tokenized_refinement["train"]]
    test_buggy_texts = [row["buggy"] for row in test_rows]

    train_embeddings = embed_codebert_texts(train_buggy_texts, codebert_tokenizer, codebert_model)
    test_embeddings = embed_codebert_texts(test_buggy_texts, codebert_tokenizer, codebert_model)

    torch.save({"meta": meta, "embeddings": train_embeddings}, RAG_TRAIN_EMBEDDINGS)
    torch.save({"meta": meta, "embeddings": test_embeddings}, RAG_TEST_EMBEDDINGS)

    del codebert_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("Using cached CodeBERT embeddings")
    train_embeddings = torch.load(RAG_TRAIN_EMBEDDINGS, map_location="cpu", weights_only=False)["embeddings"]
    test_embeddings = torch.load(RAG_TEST_EMBEDDINGS, map_location="cpu", weights_only=False)["embeddings"]

print("train embeddings:", tuple(train_embeddings.shape))
print("test embeddings:", tuple(test_embeddings.shape))

Building CodeBERT embedding caches...


CodeBERT embeddings: 100%|██████████| 205/205 [00:40<00:00,  5.04it/s]


train embeddings: (52364, 768)
test embeddings: (6545, 768)


## 17. Retrieve Top-3 Bug-fix Examples for Each Test Query

For each test buggy method, retrieve the most similar training buggy methods. The retrieved training pairs become the few-shot examples in the RAG prompt.

In [19]:
def build_or_load_topk_indices(train_embeddings, test_embeddings):
    retrieval_meta = {**rag_embedding_meta(), "top_k": RAG_TOP_K}
    if RAG_TOPK_INDICES.exists() and not FORCE_REBUILD_RAG_RETRIEVAL:
        cached = torch.load(RAG_TOPK_INDICES, map_location="cpu", weights_only=False)
        if cached.get("meta") == retrieval_meta:
            print("Using cached top-k retrieval indices")
            return cached["indices"]

    print("Computing top-k retrieval indices...")
    train_matrix = train_embeddings.to(DEVICE)
    topk_chunks = []
    query_batch_size = 256
    for start in tqdm(range(0, test_embeddings.size(0), query_batch_size), desc="Retrieving"):
        query = test_embeddings[start : start + query_batch_size].to(DEVICE)
        scores = query @ train_matrix.T
        topk = torch.topk(scores, k=RAG_TOP_K, dim=1).indices.cpu()
        topk_chunks.append(topk)
    indices = torch.cat(topk_chunks, dim=0)
    torch.save({"meta": retrieval_meta, "indices": indices}, RAG_TOPK_INDICES)
    return indices

topk_train_indices = build_or_load_topk_indices(train_embeddings, test_embeddings)
print("top-k index shape:", tuple(topk_train_indices.shape))
print("first query retrieved train rows:", topk_train_indices[0].tolist())

Computing top-k retrieval indices...


Retrieving: 100%|██████████| 26/26 [00:00<00:00, 209.48it/s]

top-k index shape: (6545, 3)
first query retrieved train rows: [9696, 21127, 31752]


## 18. Prompt Construction for Qwen

Zero-shot uses only the buggy method. RAG uses at least 3 retrieved `buggy -> fixed` examples, then asks Qwen to fix the new buggy method. The system instruction asks for only the fixed Java method so evaluation is cleaner.

In [20]:
def qwen_messages_zero_shot(buggy_method):
    return [
        {"role": "system", "content": "You are an expert Java bug fixer. Return only the fixed Java method. Do not explain."},
        {"role": "user", "content": f"Fix this buggy Java method. Return only the fixed method.\n\nBuggy method:\n{buggy_method}"},
    ]

def qwen_messages_rag(buggy_method, examples):
    chunks = ["Here are similar Java bug-fix examples."]
    for idx, example in enumerate(examples, start=1):
        chunks.append(
            f"Example {idx} buggy method:\n{example['buggy'].strip()}\n\n"
            f"Example {idx} fixed method:\n{example['fixed'].strip()}"
        )
    chunks.append(f"Now fix this buggy Java method. Return only the fixed method.\n\nBuggy method:\n{buggy_method}")
    return [
        {"role": "system", "content": "You are an expert Java bug fixer. Use the examples when helpful. Return only the fixed Java method. Do not explain."},
        {"role": "user", "content": "\n\n---\n\n".join(chunks)},
    ]

def retrieved_examples_for_test_index(test_index):
    train_rows = tokenized_refinement["train"]
    return [train_rows[int(i)] for i in topk_train_indices[test_index].tolist()]

def clean_qwen_output(text):
    text = text.strip()
    if "```" in text:
        pieces = text.split("```")
        code_pieces = [piece for piece in pieces if "public" in piece or "private" in piece or "protected" in piece or "static" in piece]
        if code_pieces:
            text = code_pieces[0]
            if text.lstrip().startswith("java"):
                text = text.lstrip()[4:]
    prefixes = ["Fixed method:", "Fixed Java method:", "Here is the fixed method:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    return text.strip()

print("Zero-shot prompt preview:")
print(qwen_messages_zero_shot(test_rows[0]["buggy"])[1]["content"][:800])
print()
print("RAG prompt preview:")
print(qwen_messages_rag(test_rows[0]["buggy"], retrieved_examples_for_test_index(0))[1]["content"][:1200])

Zero-shot prompt preview:
Fix this buggy Java method. Return only the fixed method.

Buggy method:
public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_2 . METHOD_4 ( ) ; } else if ( METHOD_3 ( VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) ) ) { return VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) ; } else { return VAR_4 . METHOD_4 ( ) ; } }


RAG prompt preview:
Here are similar Java bug-fix examples.

---

Example 1 buggy method:
private boolean METHOD_1 ( ) { VAR_1 = null ; if ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != null ? VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) : VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) ) ) ) { execute ( VAR_2 ) ; return true ; } else { VAR_2 = null ; return METHOD_7 ( ) ; } }

Example 1 fixed method:
private boolean METHOD_1 ( ) { VAR_1 = null ; if ( ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != null ? VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) : VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) )

## 19. Load Qwen Generator

This loads `Qwen/Qwen2.5-Coder-1.5B-Instruct`. On GPU it uses float16. Full test-set generation can take a while because every inference call runs a 1.5B model.

In [21]:
def load_qwen_generator():
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=dtype,
        trust_remote_code=True,
    ).to(DEVICE)
    model.eval()
    return tokenizer, model

qwen_tokenizer, qwen_model = load_qwen_generator()
print("Loaded Qwen generator on", DEVICE)

Loaded Qwen generator on cuda


## 20. Generate Qwen Predictions: Zero-shot and RAG

The JSONL files are written incrementally, so if generation stops partway through, rerun the cell and it resumes from the number of saved predictions unless `FORCE_REGENERATE_QWEN = True`.

In [23]:
def load_existing_prediction_rows(path):
    if not path.exists() or FORCE_REGENERATE_QWEN:
        return []
    return list(read_jsonl(path))

def generate_one_qwen_prediction(messages):
    prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=QWEN_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1] :]
    raw = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_qwen_output(raw)

def generate_qwen_setting(setting_name, predictions_file):
    eval_rows = test_rows if RAG_EVAL_LIMIT is None else test_rows[:RAG_EVAL_LIMIT]
    existing = load_existing_prediction_rows(predictions_file)
    start_index = len(existing)
    if start_index >= len(eval_rows):
        print(f"{setting_name}: predictions already complete at {predictions_file}")
        return existing

    mode = "a" if existing and not FORCE_REGENERATE_QWEN else "w"
    with predictions_file.open(mode, encoding="utf-8") as f:
        for test_index in tqdm(range(start_index, len(eval_rows)), desc=setting_name):
            row = eval_rows[test_index]
            if setting_name == "qwen_zero_shot":
                messages = qwen_messages_zero_shot(row["buggy"])
            elif setting_name == "qwen_rag_3shot":
                messages = qwen_messages_rag(row["buggy"], retrieved_examples_for_test_index(test_index))
            else:
                raise ValueError(setting_name)

            prediction = generate_one_qwen_prediction(messages)
            record = {"buggy": row["buggy"].strip(), "fixed": row["fixed"].strip(), "prediction": prediction}
            f.write(json.dumps(record) + "\n")
            f.flush()
            existing.append(record)
    return existing

qwen_zero_rows = generate_qwen_setting("qwen_zero_shot", QWEN_ZERO_PREDICTIONS)
qwen_rag_rows = generate_qwen_setting("qwen_rag_3shot", QWEN_RAG_PREDICTIONS)
print("zero-shot predictions:", len(qwen_zero_rows))
print("RAG predictions:", len(qwen_rag_rows))

qwen_zero_shot:   5%|▌         | 335/6319 [13:24<3:59:24,  2.40s/it]


KeyboardInterrupt: 

## 21. Score Qwen Zero-shot and RAG

Both Qwen settings are evaluated on the same test split as Pipeline A and Pipeline B.

In [ ]:
def score_prediction_jsonl(stage_name, predictions_file, metrics_file):
    if metrics_file.exists() and not FORCE_REEVALUATE_QWEN:
        metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
        print(stage_name, json.dumps(metrics, indent=2))
        return metrics

    rows = list(read_jsonl(predictions_file))
    references = [row["fixed"].strip() for row in rows]
    predictions = [row["prediction"].strip() for row in rows]
    exact_match_accuracy = sum(pred == ref for pred, ref in zip(predictions, references)) / max(1, len(rows))
    codebleu_scores = calc_codebleu(references, predictions, lang="java")
    metrics = {"stage": stage_name, "num_examples": len(rows), "exact_match_accuracy": exact_match_accuracy, **codebleu_scores}
    write_json(metrics_file, metrics)
    print(stage_name, json.dumps(metrics, indent=2))
    return metrics

qwen_zero_metrics = score_prediction_jsonl("Qwen zero-shot", QWEN_ZERO_PREDICTIONS, QWEN_ZERO_METRICS)
qwen_rag_metrics = score_prediction_jsonl("Qwen RAG 3-shot", QWEN_RAG_PREDICTIONS, QWEN_RAG_METRICS)

## 22. Final Four-way Comparison

This is the table you need for the report: Pipeline A, Pipeline B, Qwen zero-shot, and Qwen with RAG. The discussion should emphasize that the T5 models and Qwen differ in size; the comparison is about training a small task-specific model versus prompting a larger model with or without retrieval.

In [ ]:
def maybe_load_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

all_metrics = [
    maybe_load_json(PIPELINE_A_METRICS),
    maybe_load_json(PIPELINE_B_METRICS),
    maybe_load_json(QWEN_ZERO_METRICS),
    maybe_load_json(QWEN_RAG_METRICS),
]
all_metrics = [metrics for metrics in all_metrics if metrics]

print(f"{'Stage':<24} {'Examples':>8} {'Exact Match':>14} {'CodeBLEU':>10}")
print("-" * 62)
for metrics in all_metrics:
    stage = metrics.get("stage", "unknown")
    print(f"{stage:<24} {metrics.get('num_examples', 0):>8} {metrics.get('exact_match_accuracy', 0):>14.4f} {metrics.get('codebleu', 0):>10.4f}")

comparison_file = RAG_DIR / "all_pipeline_metrics_summary.json"
write_json(comparison_file, all_metrics)
print()
print("Saved summary:", comparison_file)